In [69]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (accuracy_score,f1_score,precision_score,recall_score, confusion_matrix)
import pickle


In [70]:
train = pd.read_csv("../data/processed/clean_train.csv")
test = pd.read_csv("../data/processed/clean_test.csv")

X_train = train.drop(columns=["Outcome"])
y_train = train["Outcome"]

X_test = test.drop(columns=["Outcome"])
y_test = test["Outcome"]


### GradientBoostingClassifier

In [71]:
xgbc = GradientBoostingClassifier()
xgbc.fit(X_train, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [72]:
#Rejilla de parametros
params_xgbc = {'n_estimators': [50, 100],
               'max_depth': [2, 3],
               'subsample': [0.6, 0.8],
               'min_samples_split': [10, 20],
               'min_samples_leaf': [5, 10]}

random_search_xgbc = RandomizedSearchCV(xgbc, params_xgbc, n_iter=15, cv=5, scoring="accuracy", random_state=42)
random_search_xgbc.fit(X_train, y_train)

random_search_xgbc.best_estimator_,random_search_xgbc.best_score_

(GradientBoostingClassifier(min_samples_leaf=10, min_samples_split=10,
                            n_estimators=50, subsample=0.8),
 np.float64(0.7887907705455064))

In [73]:
y_predict_train_xgbc = random_search_xgbc.predict(X_train)
y_predict_test_xgbc = random_search_xgbc.predict(X_test)

y_predict_train_xgbc, y_predict_test_xgbc

(array([0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0,
        0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0,
        1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0,
        0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1,
        1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0,
        0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1,
        0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1,
        0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 

In [74]:
print(confusion_matrix(y_train, y_predict_train_xgbc))
print(confusion_matrix(y_test, y_predict_test_xgbc))

[[364  26]
 [ 51 151]]
[[81 16]
 [23 28]]


In [75]:
def get_classifier_metrics(y_predict_test, y_test, y_predict_train, y_train, average='micro'):
    metrics_train = (accuracy_score(y_train, y_predict_train),
                     f1_score(y_train, y_predict_train, average=average),
                     precision_score(y_train, y_predict_train, average=average),
                     recall_score(y_train, y_predict_train, average=average))
    metrics_test = (accuracy_score(y_test, y_predict_test),
                    f1_score(y_test, y_predict_test, average=average),
                    precision_score(y_test, y_predict_test, average=average),
                    recall_score(y_test, y_predict_test, average=average))
    return pd.DataFrame(data=[metrics_train, metrics_test],
                        columns=['Accuracy', 'F1 Score', 'Precision', 'Recall'],
                        index=['Train set', 'Test set'])

result_xgbc = get_classifier_metrics(y_predict_test_xgbc, y_test, y_predict_train_xgbc, y_train)
result_xgbc

,Accuracy,F1 Score,Precision,Recall
Train set,0.869932,0.869932,0.869932,0.869932
Test set,0.736486,0.736486,0.736486,0.736486


### Conclusiones

-El modelo implementa el algoritmo clasico de Gradient Boosting, al usar parametros mas simples tiende ser mas conservador y menos propenso a sobreajuste lo que puede hacer que se ajuste mejor a datasets mas pequeños, esto puede explicar que devuelva mejores metricas.

### XGBClassifier

In [76]:
xgbc_optimized = XGBClassifier()
xgbc_optimized.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [77]:
params_xgbc_optimized = {'n_estimators': [50, 100, 150],
                         'learning_rate': [0.01, 0.05],
                         'max_depth': [2, 3],
                         'subsample': [0.6, 0.8],
                         'colsample_bytree': [0.6, 0.8],
                         'gamma': [0.5, 1],
                         'reg_lambda': [5, 10],
                         'reg_alpha': [1, 2],
                         'min_child_weight': [5, 10]}

random_search_xgbc_optimized = RandomizedSearchCV(xgbc_optimized, params_xgbc_optimized, n_iter=15, cv=5, scoring="accuracy", random_state=42)
random_search_xgbc_optimized.fit(X_train, y_train)
random_search_xgbc_optimized.best_estimator_,random_search_xgbc_optimized.best_score_

(XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=0.8, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric=None, feature_types=None,
               feature_weights=None, gamma=1, grow_policy=None,
               importance_type=None, interaction_constraints=None,
               learning_rate=0.05, max_bin=None, max_cat_threshold=None,
               max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
               max_leaves=None, min_child_weight=10, missing=nan,
               monotone_constraints=None, multi_strategy=None, n_estimators=150,
               n_jobs=None, num_parallel_tree=None, ...),
 np.float64(0.7786924939467312))

In [78]:
y_predict_train_xgbc_optimized = random_search_xgbc_optimized.predict(X_train)
y_predict_test_xgbc_optimized = random_search_xgbc_optimized.predict(X_test)

In [79]:
print(confusion_matrix(y_train, y_predict_train_xgbc_optimized))
print(confusion_matrix(y_test, y_predict_test_xgbc_optimized))

[[350  40]
 [ 71 131]]
[[85 12]
 [26 25]]


### Conclusiones

-Observamos una relación consistente entre los aciertos y errores tanto en el conjunto de entrenamiento (train) como en el de prueba (test). Aunque el modelo aún comete una cantidad significativa de errores, podemos descartar la presencia de data leakage, ya que los resultados son coherentes entre ambos conjuntos. Para mejorar el rendimiento del modelo y aumentar la precisión de las predicciones, sería recomendable explorar una mejor optimización de los hiperparámetros ya que este modelo suele ser mas agresivo si no se ajusta bien.

-Se suele usar en datasets mas grandes y mas complejos

In [80]:
result_xgbc_optimized = get_classifier_metrics(y_predict_test_xgbc_optimized, y_test, y_predict_train_xgbc_optimized, y_train)
result_xgbc_optimized

,Accuracy,F1 Score,Precision,Recall
Train set,0.812500,0.812500,0.812500,0.812500
Test set,0.743243,0.743243,0.743243,0.743243


### AdaBoost

In [81]:
ada = AdaBoostClassifier()
ada.fit(X_train, y_train)

,estimator,None
,n_estimators,50
,learning_rate,1.0
,algorithm,'deprecated'
,random_state,None


In [82]:
param_distributions = {
    'n_estimators': [50, 100, 200, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.5],
    'estimator': [
        DecisionTreeClassifier(max_depth=1, random_state=42),
        DecisionTreeClassifier(max_depth=2, random_state=42),
        DecisionTreeClassifier(max_depth=3, random_state=42),
    ],
    'algorithm': ['SAMME.R', 'SAMME']
}
grid_ada = RandomizedSearchCV(ada, param_distributions, cv=5, scoring='accuracy', n_jobs=-1,random_state=42)

grid_ada.fit(X_train, y_train)

print("Mejores parámetros:", grid_ada.best_params_)
print("Mejor score:", grid_ada.best_score_)

/home/vscode/.local/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/home/vscode/.local/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/home/vscode/.local/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/home/vscode/.local/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/home/vscode/.local/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning

Mejores parámetros: {'n_estimators': 400, 'learning_rate': 0.05, 'estimator': DecisionTreeClassifier(max_depth=3, random_state=42), 'algorithm': 'SAMME'}
Mejor score: 0.7719270759151119


In [83]:
y_predict_train_ada = grid_ada.predict(X_train)
y_predict_test_ada = grid_ada.predict(X_test)
y_predict_train_ada, y_predict_test_ada

(array([0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0,
        0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0,
        0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0,
        0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1,
        1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1,
        1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0,
        0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1,
        0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1,
        1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [84]:
print(confusion_matrix(y_train, y_predict_train_ada))
print(confusion_matrix(y_test, y_predict_test_ada))


[[356  34]
 [ 57 145]]
[[83 14]
 [25 26]]


In [85]:
result_ada_boost = get_classifier_metrics(y_predict_test_ada, y_test, y_predict_train_ada, y_train)
result_ada_boost

,Accuracy,F1 Score,Precision,Recall
Train set,0.846284,0.846284,0.846284,0.846284
Test set,0.736486,0.736486,0.736486,0.736486


### Conclusiones

El modelo AdaBoost sí aprende cosas útiles del dataset de diabetes, pero sus resultados son un poco peores que los otros modelos porque trabaja con árboles muy simples y es más sensible a los datos extremos o al ruido. Los modelos más modernos (como GradientBoosting, XGBoost o LightGBM) pueden captar mejor las relaciones entre variables como glucosa, edad o BMI, y por eso suelen dar métricas más altas.



In [86]:
lgbmc = LGBMClassifier()
lgbmc.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 202, number of negative: 390
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000131 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 490
[LightGBM] [Info] Number of data points in the train set: 592, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.341216 -> initscore=-0.657879
[LightGBM] [Info] Start training from score -0.657879
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [87]:
param_distributions_lgbmc = {
    'n_estimators': [200, 400, 600],          # más árboles pero con aprendizaje suave
    'learning_rate': [0.01, 0.05, 0.1],       # valores bajos para generalizar mejor
    'max_depth': [3, 5, 7],                   # limitar profundidad de los árboles
    'num_leaves': [15, 31, 63],               # pocas hojas para evitar memorizar datos
    'subsample': [0.6, 0.8],                  # usar fracción de datos en cada iteración
    'colsample_bytree': [0.6, 0.8],           # usar fracción de variables en cada árbol
    'reg_alpha': [0.1, 0.5, 1.0],             # regularización L1
    'reg_lambda': [0.1, 0.5, 1.0]}            # regularización L2

grid_lgbmc = RandomizedSearchCV(
    lgbmc,
    param_distributions=param_distributions_lgbmc,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    n_iter=30)  # limita el número de combinaciones probadas


grid_lgbmc.fit(X_train, y_train)

print("Mejores parámetros:", grid_lgbmc.best_params_)
print("Mejor score:", grid_lgbmc.best_score_)

[LightGBM] [Info] Number of positive: 161, number of negative: 312
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000052 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 420
[LightGBM] [Info] Number of data points in the train set: 473, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.340381 -> initscore=-0.661599
[LightGBM] [Info] Start training from score -0.661599
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

[LightGBM] [Info] Number of positive: 161, number of negative: 312
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000044 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 420
[LightGBM] [Info] Number of data points in the train set: 473, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.340381 -> initscore=-0.661599
[LightGBM] [Info] Start training from score -0.661599
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

In [88]:
y_predict_train_lgbmc = grid_lgbmc.predict(X_train)
y_predict_test_lgbmc = grid_lgbmc.predict(X_test)

y_predict_train_lgbmc, y_predict_test_lgbmc

(array([0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0,
        0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0,
        1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0,
        0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1,
        1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0,
        0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1,
        0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1,
        1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1,
        0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 

In [89]:
print(confusion_matrix(y_train, y_predict_train_lgbmc))
print(confusion_matrix(y_test, y_predict_test_lgbmc))


[[367  23]
 [ 39 163]]
[[80 17]
 [26 25]]


In [90]:
result_lgbmc = get_classifier_metrics(y_predict_test_lgbmc, y_test, y_predict_train_lgbmc, y_train)
result_lgbmc

,Accuracy,F1 Score,Precision,Recall
Train set,0.895270,0.895270,0.895270,0.895270
Test set,0.709459,0.709459,0.709459,0.709459


### Comparativa

In [91]:
# Comparativa de todos los modelos
comparison = pd.DataFrame({
    'Modelo': ['GradientBoosting', 'XGBoost', 'AdaBoost', 'LightGBM'],
    'Accuracy Train': [
        accuracy_score(y_train, y_predict_train_xgbc),
        accuracy_score(y_train, y_predict_train_xgbc_optimized),
        accuracy_score(y_train, y_predict_train_ada),
        accuracy_score(y_train, y_predict_train_lgbmc)
    ],
    'Accuracy Test': [
        accuracy_score(y_test, y_predict_test_xgbc),
        accuracy_score(y_test, y_predict_test_xgbc_optimized),
        accuracy_score(y_test, y_predict_test_ada),
        accuracy_score(y_test, y_predict_test_lgbmc)
    ]
})
comparison['Diferencia'] = comparison['Accuracy Train'] - comparison['Accuracy Test']
comparison.sort_values('Accuracy Test', ascending=False)

,Modelo,Accuracy Train,Accuracy Test,Diferencia
1,XGBoost,0.812500,0.743243,0.069257
0,GradientBoosting,0.869932,0.736486,0.133446
2,AdaBoost,0.846284,0.736486,0.109797
3,LightGBM,0.895270,0.709459,0.185811


### Guardado

In [92]:
with open('../models/GradientBoosting-boosting-diabetes.pkl', 'wb') as f:
    pickle.dump(xgbc, f)
with open('../models/XGBoost-boosting-diabetes.pkl', 'wb') as f:
    pickle.dump(xgbc_optimized, f)
with open('../models/ada-boosting-diabetes.pkl', 'wb') as f:
    pickle.dump(ada, f)
with open('../models/lgbmc-boosting-diabetes.pkl', 'wb') as f:
    pickle.dump(lgbmc, f)

### Conclusiones

-El modelo GradientBoosting fue el que alcanzó el mejor equilibrio entre rendimiento y generalización en el dataset de diabetes. Aunque en entrenamiento llegó a un desempeño muy alto, tras ajustar los hiperparámetros logró mantener una precisión cercana al 75% en el conjunto de prueba, superando a otros algoritmos como AdaBoost o XGBoost. Esto muestra que GradientBoosting es capaz de capturar mejor las relaciones no lineales entre variables clínicas como glucosa, BMI y edad, ofreciendo un modelo más robusto y competitivo para este tipo de problemas.

# Conclusiones Generales

AdaBoost:

-Funciona combinando clasificadores débiles (normalmente árboles muy simples) y ajustando los pesos de las muestras mal clasificadas en cada iteración.
-Es más sencillo de configurar, pero también más sensible al ruido y a valores extremos.
-Se aplica mejor en datasets pequeños y relativamente limpios, donde la simplicidad ayuda a evitar sobreajuste.

GradientBoosting:

-Construye árboles de forma secuencial, corrigiendo los errores del anterior mediante gradiente descendente.
-Tiene más hiperparámetros que AdaBoost y suele generalizar mejor.
-Es útil en datasets tabulares de tamaño medio, donde se busca un buen equilibrio entre interpretabilidad y rendimiento.

XGBoost:

-Es una versión optimizada de GradientBoosting, con regularización avanzada y técnicas de subsampling.
-Es muy eficiente y robusto frente a ruido, por lo que suele rendir mejor en datasets grandes y complejos.
-Se aplica en competiciones de machine learning y problemas con alta dimensionalidad, como texto o datos financieros.

LightGBM:

-Utiliza un crecimiento “leaf-wise” en lugar de “level-wise”, lo que le permite entrenar más rápido y con mayor precisión.
-Es muy potente, pero puede sobreajustar si no se controlan parámetros como num_leaves y max_depth.
-Se recomienda para datasets grandes y con muchas variables, donde la velocidad y la capacidad de capturar relaciones no lineales son críticas.

En este análisis hemos ajustado los hiperparámetros de todos los modelos, lo que naturalmente influye en sus resultados y puede hacer que algunos destaquen más que otros. Para tener una comparación más objetiva de su comportamiento, sería recomendable evaluar primero cada algoritmo con sus configuraciones por defecto y luego contrastar esos resultados con los obtenidos tras la optimización.